In [1]:
import os
import numpy as np
from itertools import combinations
from scipy.stats import wilcoxon
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multitest import multipletests

In [2]:
def get_model_metrics(file_path, model_name):
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    current_model = None
    header = None
    data = []
    
    for line in lines:
        if line.startswith("Result for"):
            current_model = line.replace("Result for ", "").strip()
        elif line.startswith("Subject,"):
            header = line.split(",")
        elif line.startswith("chb") and current_model == model_name:
            data.append(line.split(","))

    df = pd.DataFrame(data, columns=header)

    metric_cols = [c for c in df.columns if c != "Subject"]
    df[metric_cols] = df[metric_cols].astype(float)

    
    metrics_dict = {
        metric: df[metric].tolist()
        for metric in metric_cols
    }

    return metrics_dict

In [3]:
def rank_biserial_wilcoxon(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    diff = x - y
    diff_nonzero = diff[diff != 0]

    if len(diff_nonzero) == 0:
        return 0.0
    abs_diff = np.abs(diff_nonzero)
    ranks = pd.Series(abs_diff).rank(method='average').to_numpy()

    W_plus = np.sum(ranks[diff_nonzero > 0])
    W_minus = np.sum(ranks[diff_nonzero < 0])
    r_rb = (W_plus - W_minus) / (W_plus + W_minus)

    return r_rb


def calculate_wilcoxon_results(results_path, method1s, method2, metrics, output_path='wilcoxon_results.csv'):
    models = ['Random Forest', 'XGBoost', 'SVM']
    all_results = []  

    for method1 in method1s:
        model_scores = {}
        for model in models:
            dict_scores = {}
            for file in os.listdir(results_path):
                method = file.split('_')[0]
                if method in [method1, method2]:
                    full_path = os.path.join(results_path, file)
                    dict_scores[method] = get_model_metrics(full_path, model)

            if method1 not in dict_scores:
                raise ValueError(f'Không tìm thấy dữ liệu cho method: {method1}')
            if method2 not in dict_scores:
                raise ValueError(f'Không tìm thấy dữ liệu cho method: {method2}')

            model_scores[model] = dict_scores

        
        for metric in metrics:
            metric_rows = []
            p_values = [] 

            for model in models:
                dict_scores = model_scores[model]

                scores1 = np.asarray(dict_scores[method1][metric], dtype=float)
                scores2 = np.asarray(dict_scores[method2][metric], dtype=float)

                difference = np.mean(scores1 - scores2)
                stat, p = wilcoxon(scores1, scores2)
                r_rb = rank_biserial_wilcoxon(scores1, scores2)

                p_values.append(p)
                metric_rows.append({
                    'Model': model,
                    'Method 1': method1,
                    'Method 2': method2,
                    'Metric': metric,
                    'Difference': difference,
                    'p-value': p,
                    'r_rb': r_rb
                })

            # Holm correction 
            reject, p_holm, _, _ = multipletests(p_values, method='holm')
            for row, p_adj in zip(metric_rows, p_holm):
                row['p_Holm'] = p_adj

            all_results.extend(metric_rows)

    df_results = pd.DataFrame(all_results)
    df_results = df_results[[
            'Model',
            'Method 1',
            'Method 2',
            'Metric',
            'Difference',
            'p_Holm',
            'r_rb'
        ]]
    df_results.to_csv(output_path, index=False, encoding='utf-8')
    return df_results

In [4]:
calculate_wilcoxon_results('/kaggle/input/datasets/ngoclannguyen00/metrics-results', ['PLI-Trad', 'wPLI-Trad'], 'Traditional FTs', ['Balanced accuracy', 'MCC'])

,Model,Method 1,Method 2,Metric,Difference,p_Holm,r_rb
0,Random Forest,PLI-Trad,Traditional FTs,Balanced accuracy,0.036001,0.009614,0.838235
1,XGBoost,PLI-Trad,Traditional FTs,Balanced accuracy,0.026116,0.097989,0.470588
2,SVM,PLI-Trad,Traditional FTs,Balanced accuracy,0.042916,0.039942,0.661765
3,Random Forest,PLI-Trad,Traditional FTs,MCC,0.111840,0.004039,0.911765
4,XGBoost,PLI-Trad,Traditional FTs,MCC,0.098609,0.010468,0.735294
5,SVM,PLI-Trad,Traditional FTs,MCC,0.130366,0.010468,0.794118
6,Random Forest,wPLI-Trad,Traditional FTs,Balanced accuracy,0.036159,0.006847,0.867647
7,XGBoost,wPLI-Trad,Traditional FTs,Balanced accuracy,0.025035,0.108941,0.455882
8,SVM,wPLI-Trad,Traditional FTs,Balanced accuracy,0.048222,0.010468,0.794118
9,Random Forest,wPLI-Trad,Traditional FTs,MCC,0.112091,0.004039,0.911765
